# The compounder screen — residual Sharpe x ROE spread (H.89.6)

**[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/quality_overlay_screen.ipynb)** · **[Get API key](https://riskmodels.app/get-key)**

A **quality overlay**: rank a watchlist on two independent axes and keep only names strong on both —

1. **Residual Sharpe** (`residual_return` from `get_lstar`) — risk-adjusted *skill-based* alpha, net of market/sector/subsector beta. High residual Sharpe means the stock's excess return isn't just a levered market/sector bet.
2. **ROE spread** (`roe_ttm - cost_of_equity` from `get_fundamentals`) — the equity-charge form of economic profit, normalized to a percentage so it's comparable across market caps (the dollar `economic_profit` field scales with `total_equity`, which isn't in the derived-only response — see `CLAUDE.md` "Derived-only licensing").

A stock high on **both** axes is generating genuine idiosyncratic return *and* earning more than its cost of capital — the two ingredients of a durable compounder, as opposed to a name that's merely riding a factor tailwind or growing revenue below its cost of capital.


### Colab only (skip locally)

Run once to install the SDK; local users should `pip install riskmodels-py` in their venv.


In [1]:
import sys

try:
    import google.colab  # noqa: F401
    _COLAB = True
except ImportError:
    _COLAB = False

if _COLAB:
    import subprocess

    _deps = ["python-dotenv"]
    _pypi = "riskmodels-py>=0.3.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: use your environment (pip install riskmodels-py python-dotenv).")


Local: use your environment (pip install riskmodels-py python-dotenv).


## 1. Connect — `RiskModelsClient.from_env()`

Loads **`RISKMODELS_API_KEY`** from shell env, `.env` / `.env.local` (when `python-dotenv` is installed), or Colab Secrets.


In [2]:
import os

from IPython.display import display

from riskmodels import RiskModelsClient
from riskmodels.client import DEFAULT_BASE_URL
from riskmodels.notebook import load_notebook_dotenv

load_notebook_dotenv()
print("RISKMODELS_BASE_URL =", os.environ.get("RISKMODELS_BASE_URL", DEFAULT_BASE_URL))
client = RiskModelsClient.from_env()


RISKMODELS_BASE_URL = https://riskmodels.app/api


## 2. Watchlist

A deliberately mixed set — mega-cap tech, staples, industrials — so the screen has to discriminate, not just sort by sector momentum.


In [3]:
WATCHLIST = ["AAPL", "MSFT", "NVDA", "COST", "WMT", "XOM", "JNJ", "CAT"]


## 3. Residual Sharpe per ticker

`get_ticker_returns` carries `l3_residual_er` (an explained-variance *fraction*, 0-1) but no residual *return* series. `get_lstar(years=2)` is the SDK method that does: for each date it dispatches to the simplest hedge level (L1/L2/L3) whose marginal explained-return clears a threshold, then returns that level's daily `residual_return`. Annualized Sharpe over the full window — deliberately simple (no rolling window) since this is a screen, not a timing signal.


In [4]:
import numpy as np


def residual_sharpe(ticker: str, years: int = 2) -> float | None:
    df = client.get_lstar(ticker, years=years)
    r = df.get("residual_return")
    if r is None or r.dropna().empty:
        return None
    r = r.dropna()
    if r.std() == 0:
        return None
    return float(r.mean() / r.std() * np.sqrt(252))


sharpe_by_ticker = {t: residual_sharpe(t) for t in WATCHLIST}
sharpe_by_ticker


{'AAPL': 0.10196404620243678,
 'MSFT': -1.1589712909271774,
 'NVDA': 0.1561551326870415,
 'COST': 0.007010124192302697,
 'WMT': 0.9117525670385396,
 'XOM': 0.57178486708075,
 'JNJ': 2.373279020751213,
 'CAT': 1.4626542729392005}

## 4. ROE spread per ticker

Latest PIT-visible quarter's `roe_ttm - cost_of_equity` from `get_fundamentals`. `cost_of_equity` uses the caller-supplied `erp` (default 0.05) and defaults to the `10y` risk-free tenor — see the endpoint's `disclosures.conditional_beta_cost_of_equity` caveat: `beta_market` is a short-half-life conditional beta, not a textbook CAPM beta, so `cost_of_equity` can occasionally sit below `rf_rate` for defensive names. That's a property of the beta, not a data error.


In [5]:
def roe_spread(ticker: str) -> tuple[float | None, float | None]:
    df = client.get_fundamentals(ticker, as_dataframe=True)
    if df.empty:
        return None, None
    last = df.iloc[-1]
    roe, coe = last["roe_ttm"], last["cost_of_equity"]
    if roe is None or coe is None:
        return None, None
    return float(roe - coe), float(last.get("economic_profit") or float("nan"))


spread_by_ticker = {t: roe_spread(t) for t in WATCHLIST}
spread_by_ticker


{'AAPL': (1.3738337461419186, 146300933417.04025),
 'MSFT': (0.24706169187654234, 102374209690.20303),
 'NVDA': (0.9866400423699141, 192862473495.28784),
 'COST': (0.24036301881520047, 8054324197.496521),
 'WMT': (0.18865179213845393, 17795524204.40095),
 'XOM': (0.04214026095877305, 10719681666.317137),
 'JNJ': (0.21656175888664214, 17581782263.9733),
 'CAT': (0.3408983117807388, 6361503112.512972)}

## 5. Combine + rank

Cross-sectional rank on each axis (percentile), then average — no arbitrary weighting scheme, just "strong on both" vs. "strong on one, weak on the other" vs. "weak on both". This is a screen to generate a short list for further diligence, not a standalone signal.


In [6]:
import pandas as pd

rows = []
for t in WATCHLIST:
    rs = sharpe_by_ticker.get(t)
    spread, econ_profit_dollar = spread_by_ticker.get(t, (None, None))
    rows.append(
        {
            "ticker": t,
            "residual_sharpe": rs,
            "roe_spread": spread,
            "economic_profit_latest_quarter_$": econ_profit_dollar,
        }
    )
screen = pd.DataFrame(rows).dropna(subset=["residual_sharpe", "roe_spread"])
screen["sharpe_pct"] = screen["residual_sharpe"].rank(pct=True)
screen["spread_pct"] = screen["roe_spread"].rank(pct=True)
screen["compounder_score"] = (screen["sharpe_pct"] + screen["spread_pct"]) / 2
screen = screen.sort_values("compounder_score", ascending=False).reset_index(drop=True)
display(screen)


,ticker,residual_sharpe,roe_spread,economic_profit_latest_quarter_$,sharpe_pct,spread_pct,compounder_score
0,CAT,1.462654,0.340898,6.361503e+09,0.875,0.750,0.8125
1,AAPL,0.101964,1.373834,1.463009e+11,0.375,1.000,0.6875
2,NVDA,0.156155,0.986640,1.928625e+11,0.500,0.875,0.6875
3,JNJ,2.373279,0.216562,1.758178e+10,1.000,0.375,0.6875
4,WMT,0.911753,0.188652,1.779552e+10,0.750,0.250,0.5000
5,MSFT,-1.158971,0.247062,1.023742e+11,0.125,0.625,0.3750
6,COST,0.007010,0.240363,8.054324e+09,0.250,0.500,0.3750
7,XOM,0.571785,0.042140,1.071968e+10,0.625,0.125,0.3750


## Next steps

- Widen `WATCHLIST` with `riskmodels_search_tickers` / `client.search_tickers()` for a sector or the full universe (subject to per-call fundamentals billing — no batch/bulk variant exists by design, see `OPENAPI_SPEC.yaml` `/fundamentals/{ticker}`).
- Pair with `client.get_fundamentals_sensitivity_grid(ticker)` to see how sensitive a borderline name's `roe_spread` is to the ERP assumption before acting on the screen.
- **Pre-earnings hedge timing:** [`pre_earnings_hedge_timing.ipynb`](./pre_earnings_hedge_timing.ipynb)
